# LDA Final Project Notebook

This notebook completes the Final Project LDA deliverables using the class LDA-with-sklearn notebook as the main pattern. It builds constitution-level document strings from the parsed `CORPUS`, vectorizes those strings with `CountVectorizer`, fits sklearn `LatentDirichletAllocation`, and saves the required `TOPIC`, `THETA`, and `PHI` tables.


In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation as LDA, PCA

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
data_home = "."
output_dir = Path("output")
fig_dir = Path("iframe_figures")
fig_dir.mkdir(exist_ok=True)

data_prefix = "constitutions"
CSV_DELIM = "|"

OHCO = [
    "constitution_id",
    "provision_num",
    "para_num",
    "sent_num",
    "token_num",
]

bags = dict(
    SENTS=OHCO[:4],
    PARAS=OHCO[:3],
    PROVISIONS=OHCO[:2],
    CONSTITUTIONS=OHCO[:1],
)

bag = "CONSTITUTIONS"
BAG = bags[bag]

n_topics = 5
max_iter = 100
n_top_terms = 12
random_state = 42
TNAMES = [f"T{x:02d}" for x in range(n_topics)]

## Load Tables

The source tables come from the earlier project notebooks and use the same pipe delimiter as the rest of the final project output.


In [ ]:
LIB = pd.read_csv(
    output_dir / f"{data_prefix}-LIB.csv",
    sep=CSV_DELIM,
    keep_default_na=False,
).set_index("constitution_id")

TOKEN = pd.read_csv(
    output_dir / f"{data_prefix}-CORPUS.csv",
    sep=CSV_DELIM,
    keep_default_na=False,
).set_index(OHCO)

VOCAB_SOURCE = pd.read_csv(
    output_dir / f"{data_prefix}-VOCAB.csv",
    sep=CSV_DELIM,
    keep_default_na=False,
).set_index("term_str")

LIB.head()

## Build LDA Documents

Following the class LDA example, the token table is grouped into document strings and passed to `CountVectorizer`. For this corpus, the bag is the full constitution because the final project PCA tables are also constitution-level.

Filtering choices:

- Keep nouns and verbs only (`pos_group` in `NN`, `VB`).
- Drop terms already marked as stopwords in `VOCAB`.
- Drop terms shorter than three characters.
- Add a few corpus-structure words to sklearn's English stopword list.


In [ ]:
TOKEN = TOKEN.dropna(subset=["term_str"]).copy()
TOKEN = TOKEN.loc[TOKEN.term_str.astype(str).str.len() > 0]
TOKEN["term_str"] = TOKEN.term_str.astype(str)
TOKEN["pos_group"] = TOKEN.pos_group.astype(str)

VOCAB_SOURCE["stop"] = pd.to_numeric(VOCAB_SOURCE["stop"], errors="coerce").fillna(0).astype(int)
VOCAB_SOURCE["n_chars"] = pd.to_numeric(VOCAB_SOURCE["n_chars"], errors="coerce").fillna(0).astype(int)
valid_terms = VOCAB_SOURCE[(VOCAB_SOURCE.stop == 0) & (VOCAB_SOURCE.n_chars >= 3)].index

lda_tokens = TOKEN[
    TOKEN.pos_group.isin(["NN", "VB"])
    & TOKEN.term_str.isin(valid_terms)
].copy()

DOCS = lda_tokens.groupby(BAG).term_str.apply(lambda x: " ".join(map(str, x))).to_frame("doc_str")
DOCS = DOCS.reindex(LIB.index).fillna({"doc_str": ""})
DOCS.head()

In [ ]:
my_stop_words = sorted(text.ENGLISH_STOP_WORDS.union({
    "shall",
    "may",
    "article",
    "section",
    "chapter",
    "part",
    "constitution",
    "law",
    "laws",
    "act",
    "acts",
}))

count_engine = CountVectorizer(
    max_df=.9,
    min_df=5,
    stop_words=my_stop_words,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]{2,}\b",
)

count_model = count_engine.fit_transform(DOCS.doc_str)
TERMS = count_engine.get_feature_names_out()

DTM = pd.DataFrame(count_model.toarray(), index=DOCS.index, columns=TERMS)
DTM.index.name = "constitution_id"
DTM.shape

## Generate LDA Model

This section mirrors the class example: create the sklearn engine, fit the count matrix, then convert the model arrays into `THETA` and `PHI` DataFrames.


In [ ]:
topic_engine = LDA(
    n_components=n_topics,
    max_iter=max_iter,
    learning_method="batch",
    random_state=random_state,
)

topic_model = topic_engine.fit_transform(count_model)

THETA = pd.DataFrame(topic_model, index=DOCS.index, columns=TNAMES)
THETA.index.name = "constitution_id"
THETA.columns.name = "topic_id"

PHI = pd.DataFrame(topic_engine.components_, columns=TERMS, index=TNAMES)
PHI.index.name = "topic_id"
PHI.columns.name = "term_str"

THETA.head()

In [ ]:
PHI.iloc[:, :10]

## Create `TOPIC` Table

The `TOPIC` table contains the top topic terms, document weight summaries, and a best-guess label for each topic. The topics are sorted by mean document weight so the five-topic deliverable can be read directly from the table.


In [ ]:
TOPIC = PHI.stack().groupby("topic_id")    .apply(lambda x: " ".join(x.sort_values(ascending=False).head(n_top_terms).reset_index().term_str))    .to_frame("top_terms")

TOPIC["top_5_terms"] = PHI.stack().groupby("topic_id")    .apply(lambda x: " ".join(x.sort_values(ascending=False).head(5).reset_index().term_str))
TOPIC["doc_weight_sum"] = THETA.sum(axis=0)
TOPIC["doc_weight_mean"] = THETA.mean(axis=0)
TOPIC["term_freq"] = PHI.sum(axis=1) / PHI.sum(axis=1).sum()
TOPIC = TOPIC.sort_values("doc_weight_mean", ascending=False)

label_map = {
    TOPIC.index[0]: "State authority and public administration",
    TOPIC.index[1]: "Rights, courts, and legal protections",
    TOPIC.index[2]: "Finance, taxation, and public funds",
    TOPIC.index[3]: "Elections and representative offices",
    TOPIC.index[4]: "Local government and citizenship",
}
TOPIC["label"] = TOPIC.index.map(label_map)

TOPIC[["label", "top_5_terms", "doc_weight_mean", "term_freq"]]

## LDA TOPIC Deliverable

- UVA Box URL: add after uploading `output/constitutions-LDA_TOPIC-CONSTITUTIONS.csv`
- UVA Box URL of count matrix used to create: add after uploading `output/constitutions-LDA_DTM-CONSTITUTIONS.csv`
- GitHub URL for notebook used to create: `LDA_FinalProject.ipynb`
- Delimitter: `|`
- Libary used to compute: `sklearn.decomposition.LatentDirichletAllocation`
- A description of any filtering, e.g. POS: nouns and verbs only; removed VOCAB stopwords; removed terms shorter than three characters; `CountVectorizer(max_df=.9, min_df=5)` with sklearn English stopwords plus corpus structure terms.
- Number of components: `5`
- Any other parameters used: `max_iter=100`, `learning_method='batch'`, `random_state=42`, constitution-level bag.
- Top 5 words and best-guess labels for topic five topics by mean document weight:
  - T04: republic national constitutional deputies prime — State authority and public administration
  - T02: national parliament federal house committee — Rights, courts, and legal protections
  - T00: parliament commission subsection house governor — Finance, taxation, and public funds
  - T01: congress services federal national tribunal — Elections and representative offices
  - T03: parliament clause commission union chief — Local government and citizenship


## Explore Topic Weights

The bar plot shows average topic prevalence across constitution documents.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_topic = TOPIC.sort_values("doc_weight_mean", ascending=True)
ax.barh(plot_topic["label"], plot_topic["doc_weight_mean"], color="#4c78a8")
ax.set_xlabel("Mean document weight")
ax.set_ylabel("")
ax.set_title("LDA Topic Prevalence")
for i, (_, row) in enumerate(plot_topic.iterrows()):
    ax.text(row.doc_weight_mean + 0.003, i, row.top_5_terms, va="center", fontsize=8)
fig.tight_layout()
fig.savefig(fig_dir / "lda_topic_prevalence.png", dpi=180, bbox_inches="tight")

![](iframe_figures/lda_topic_prevalence.png)

## LDA THETA Deliverable

- UVA Box URL: add after uploading `output/constitutions-LDA_THETA-CONSTITUTIONS.csv`
- GitHub URL for notebook used to create: `LDA_FinalProject.ipynb`
- Delimitter: `|`

`THETA` has one row per constitution and one column per topic. Each row sums to 1, so each value is the estimated topic weight for that constitution.


In [ ]:
THETA.head()

In [ ]:
THETA.sum(axis=1).describe()

## LDA PHI Deliverable

- UVA Box URL: add after uploading `output/constitutions-LDA_PHI-CONSTITUTIONS.csv`
- GitHub URL for notebook used to create: `LDA_FinalProject.ipynb`
- Delimitter: `|`

`PHI` has one row per topic and one column per term. Each value is sklearn's learned topic-term component weight.


In [ ]:
PHI.iloc[:, :20]

## LDA + PCA Visualization

Apply PCA to the `THETA` table and plot documents in the space opened by the first two components. Point size is based on the mean document weight of each document's dominant topic, and color is based on the `original_year` metadata field from `LIB`.


In [ ]:
THETAX = THETA.join(LIB, how="left")
THETAX["dominant_topic"] = THETA.idxmax(axis=1)
THETAX["dominant_topic_weight"] = THETA.max(axis=1)
THETAX["dominant_topic_label"] = THETAX.dominant_topic.map(TOPIC.label)
THETAX["dominant_topic_mean_weight"] = THETAX.dominant_topic.map(TOPIC.doc_weight_mean)

pca_engine = PCA(n_components=2, random_state=random_state)
TPC = pd.DataFrame(pca_engine.fit_transform(THETA), index=THETA.index, columns=["PC0", "PC1"])
TPC.index.name = "constitution_id"
TPC = TPC.join(THETAX[[
    "country",
    "file_year",
    "original_year",
    "n_chars",
    "dominant_topic",
    "dominant_topic_label",
    "dominant_topic_weight",
    "dominant_topic_mean_weight",
]])

TPC.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
year = pd.to_numeric(TPC["original_year"], errors="coerce").fillna(pd.to_numeric(TPC["file_year"], errors="coerce"))
sizes = 1600 * TPC.dominant_topic_mean_weight
sc = ax.scatter(
    TPC.PC0,
    TPC.PC1,
    c=year,
    s=sizes,
    alpha=.78,
    cmap="viridis",
    edgecolor="#333333",
    linewidth=.4,
)
for topic_id, group in TPC.groupby("dominant_topic"):
    centroid = group[["PC0", "PC1"]].mean()
    label = f"{topic_id}\n{TOPIC.loc[topic_id, 'label']}"
    ax.text(centroid.PC0, centroid.PC1, label, weight="bold", fontsize=8, ha="center")
cb = fig.colorbar(sc, ax=ax)
cb.set_label("Original year")
ax.set_title("Documents in PCA Space Derived from LDA THETA")
ax.set_xlabel("PC0")
ax.set_ylabel("PC1")
fig.tight_layout()
fig.savefig(fig_dir / "lda_theta_pca.png", dpi=180, bbox_inches="tight")

![](iframe_figures/lda_theta_pca.png)

Interpretation: the PCA projection of `THETA` mainly separates constitutions by their dominant topic mixture rather than by a clean chronological gradient. Documents with strong weights on the larger state-administration and rights/legal topics cluster toward the central mass, while smaller topic mixtures form lighter edge groups. The year color is mixed across the plot, suggesting that topic structure in this constitution corpus is not explained by original year alone.


## Save LDA Outputs

The files below are the LDA deliverable tables and supporting model artifacts.


In [ ]:
save_path = output_dir / data_prefix

DOCS.to_csv(f"{save_path}-LDA_DOCS-{bag}.csv", sep=CSV_DELIM)
DTM.to_csv(f"{save_path}-LDA_DTM-{bag}.csv", sep=CSV_DELIM)
TOPIC.to_csv(f"{save_path}-LDA_TOPIC-{bag}.csv", sep=CSV_DELIM)
TOPIC.to_csv(f"{save_path}-LDA_TOPICS-{bag}.csv", sep=CSV_DELIM)
THETA.to_csv(f"{save_path}-LDA_THETA-{bag}.csv", sep=CSV_DELIM)
PHI.to_csv(f"{save_path}-LDA_PHI-{bag}.csv", sep=CSV_DELIM)
TPC.to_csv(f"{save_path}-LDA_THETA_PCA-{bag}.csv", sep=CSV_DELIM)

with open(output_dir / f"{data_prefix}-LDA-topic_engine-{bag}.pickle", "wb") as f:
    pickle.dump(topic_engine, f)
with open(output_dir / f"{data_prefix}-LDA-count_engine-{bag}.pickle", "wb") as f:
    pickle.dump(count_engine, f)
with open(output_dir / f"{data_prefix}-LDA-count_model-{bag}.pickle", "wb") as f:
    pickle.dump(count_model, f)

sorted(output_dir.glob(f"{data_prefix}-LDA*{bag}*"))

## THETA Heatmap

This extra figure is useful for checking that the document-topic matrix has real variation and that the LDA model is not assigning every document the same topic mix.


In [ ]:
heat = THETA.join(THETAX[["dominant_topic"]]).sort_values("dominant_topic").drop(columns=["dominant_topic"])
fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(heat.values, aspect="auto", cmap="YlGnBu")
ax.set_title("Document Topic Weights (THETA)")
ax.set_xlabel("Topic")
ax.set_ylabel("Constitution")
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns)
ax.set_yticks([])
fig.colorbar(im, ax=ax, label="topic weight")
fig.tight_layout()
fig.savefig(fig_dir / "lda_theta_heatmap.png", dpi=180, bbox_inches="tight")

![](iframe_figures/lda_theta_heatmap.png)